# SymFormer / TBX11K — Colab Runbook (Core-Method PoC)

Executes the phases in `plan.md` on a free Colab **T4**. Run cells top-to-bottom.

**Stack: torch + torchvision only.** No mmcv/mmdetection — OpenMMLab ships wheels only up to
~torch 2.1 / Python 3.11 and Colab is on Python 3.12, so `mim install mmcv` falls into a failing
source build. torchvision gives the same ResNet-50 + FPN + RetinaNet architecture with **zero
installs**. Our SAS block is pure torch and is unchanged.

**Storage split** (free Drive is 15GB; the Colab VM has ~100GB of ephemeral disk):

| What | Where | Size |
|---|---|---|
| code / docs | **GitHub** → `/content/FOP` | ~200 KB |
| raw TBX11K | **`/content`** (ephemeral — never Drive) | ~tens of GB |
| compact TB-only 512² dataset | **Drive** | ~250–400 MB |
| checkpoints + logs | **Drive** | ~600 MB (+~300 MB transient per ablation cell) |

Training cells auto-resume, so after a time-out just re-run the same cell.
See `paper.md` (method + target numbers), `CLAUDE.md` (scope/recipe), `limitations.md` (caveats).

## Phase 1 — environment

In [ ]:
!nvidia-smi   # confirm a GPU (ideally a T4) is attached; if none, change runtime or retry later

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
# ---- paths ----
# Free Drive is only 15GB, but the Colab VM has ~100GB of EPHEMERAL local disk, so:
#   raw TBX11K (tens of GB) -> /content  (download, prep, then let it die with the session)
#   compact 512 dataset + checkpoints/logs -> Drive (must survive time-outs)
# Expected Drive usage: ~1.5GB peak.
REPO_URL = 'https://github.com/Manas-Maahir/FOP.git'
PROJECT  = '/content/FOP'
DRIVE    = '/content/drive/MyDrive'
DATA_RAW = '/content/TBX11K'              # raw TBX11K — EPHEMERAL (fixed up after extraction)
DATA_512 = f'{DRIVE}/tbx11k_tb512'        # compact TB-only 512 dataset (built in Phase 3)
RUNS     = f'{DRIVE}/tb_runs'             # checkpoints + logs (persist across sessions)
os.makedirs(RUNS, exist_ok=True)
print('project       :', PROJECT)
print('raw (temp)    :', DATA_RAW)
print('compact(Drive):', DATA_512)
print('runs (Drive)  :', RUNS)

In [ ]:
# Pull the code from GitHub. Re-running this in a later session picks up anything pushed since.
if os.path.isdir(PROJECT):
    !cd {PROJECT} && git pull --ff-only
else:
    !git clone {REPO_URL} {PROJECT}
%cd {PROJECT}
!ls

In [ ]:
# No mmcv/mmdet. torch + torchvision ship with Colab; pycocotools usually does too.
# DO NOT `pip install -U openmim` — it downgrades setuptools and breaks pkg_resources on Py 3.12.
import torch, torchvision, sys
print('python     ', sys.version.split()[0])
print('torch      ', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')
print('torchvision', torchvision.__version__)
try:
    import pycocotools; print('pycocotools OK')
except ImportError:
    !pip install -q pycocotools
    import pycocotools; print('pycocotools installed')
open('requirements.lock.txt','w').write(
    f"python=={sys.version.split()[0]}\ntorch=={torch.__version__}\n"
    f"torchvision=={torchvision.__version__}\n")
print('wrote requirements.lock.txt  (RECORD these in results.md)')

## Phase 6 (early) — unit tests
These gate the model code and need **no data** — run them before downloading tens of GB.

In [ ]:
!python tests/test_numpy_ref.py    # mirror + PE math (torch-free)
!python tests/test_sas.py          # the 4 required SAS tests + numpy cross-checks
!python tests/test_tv_model.py     # RetinaNet + SAS wiring, weight sharing, train/infer

## Phase 2 — download the raw dataset
Goes to **ephemeral `/content`**, never Drive. Skipped automatically if the compact set already
exists on Drive from a previous session.

If `gdown` fails (Google Drive quota on large public files is common), download the archive
manually from https://github.com/yun-liu/Tuberculosis and upload/extract it to `/content`.

In [ ]:
import os, glob, shutil

def compact_ready(p):
    """True only if the compact dataset REALLY exists — never trust a bare directory.
    A failed earlier run can leave empty dirs behind that otherwise look 'done'."""
    return (os.path.isfile(f'{p}/annotations/tb_train_agnostic.json')
            and os.path.isfile(f'{p}/annotations/tb_val_agnostic.json')
            and os.path.isdir(f'{p}/images/train')
            and len(os.listdir(f'{p}/images/train')) > 0)

# --- clean up stale leftovers from any previous failed run ---
if os.path.isdir(DATA_512) and not compact_ready(DATA_512):
    print('removing stale/incomplete compact dir:', DATA_512)
    shutil.rmtree(DATA_512, ignore_errors=True)
if os.path.isdir(DATA_RAW) and not os.listdir(DATA_RAW):
    print('removing empty leftover raw dir:', DATA_RAW)
    os.rmdir(DATA_RAW)

if compact_ready(DATA_512):
    print('compact dataset already on Drive ->', DATA_512)
    print('Nothing to download. Skip to Phase 4.')
else:
    have_raw = os.path.isdir(DATA_RAW) and os.listdir(DATA_RAW)
    if not have_raw:
        print('Downloading TBX11K (tens of GB) to ephemeral /content ...')
        !pip install -q -U gdown
        # official link from https://github.com/yun-liu/Tuberculosis
        !gdown --fuzzy 'https://drive.google.com/file/d/1r-oNYTPiPCOUzSjChjCIYTdkjBTugqxR/view?usp=sharing' -O /content/tbx11k.zip
        if os.path.isfile('/content/tbx11k.zip'):
            print('unzipping ...')
            !unzip -q /content/tbx11k.zip -d /content && rm -f /content/tbx11k.zip
        else:
            print('\n!! gdown produced no archive — Google Drive quota on large public files is'
                  '\n   a common failure. Download the archive manually from'
                  '\n   https://github.com/yun-liu/Tuberculosis and extract it under /content,'
                  '\n   then re-run this cell.')
    print('\nTop-level dirs under /content (find the extracted dataset root):')
    for d in sorted(glob.glob('/content/*')):
        if os.path.isdir(d) and os.path.basename(d) not in ('drive', 'sample_data', 'FOP'):
            try:
                n = len(os.listdir(d))
            except OSError:
                n = -1
            print(f'    {d}   ({n} entries)')
!df -h /content | tail -1

## Phase 2 GATE — inspect the actual layout
The official README documents the download links but **not** the folder structure, so we discover
it rather than assume it. Check the report: XML count, image count, and that every class name maps
to active/latent. Expect ~1,200 TB images (train ~600 / val ~200 per paper Table 2).

In [ ]:
# If the listing above shows a different folder name, point DATA_RAW at it here.
DATA_RAW = '/content/TBX11K'
!python tools/prepare_tbx11k.py --inspect --src {DATA_RAW}

## Phase 3 — build the compact TB-only 512² COCO set → Drive
Layout is auto-discovered. Override with `--xml-dir` / `--train-list` / `--val-list` if the report
above shows something unexpected. This is the only thing that gets persisted.

In [ ]:
!python tools/prepare_tbx11k.py --selftest    # verify the prep logic first (synthetic)
!python tools/prepare_tbx11k.py --src {DATA_RAW} --dst {DATA_512} --size 512 --write-agnostic
!du -sh {DATA_512}   # expect a few hundred MB

In [ ]:
# GATE: sanity-check a few overlays before training on this data.
import json, matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image
d = json.load(open(f'{DATA_512}/annotations/tb_train.json'))
by_img = {}
for a in d['annotations']:
    by_img.setdefault(a['image_id'], []).append(a)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, im in zip(axes, d['images'][:4]):
    ax.imshow(Image.open(f"{DATA_512}/images/train/{im['file_name']}"), cmap='gray')
    for a in by_img.get(im['id'], []):
        x, y, w, h = a['bbox']
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor='lime', lw=2))
    ax.set_title(f"{im['file_name'][:18]}  ({len(by_img.get(im['id'], []))} box)"); ax.axis('off')
plt.tight_layout(); plt.savefig('/content/overlay_check.png', dpi=70)
from IPython.display import Image as IPImage, display; display(IPImage('/content/overlay_check.png'))
print(f"train images={len(d['images'])} boxes={len(d['annotations'])}")

## Phase 4 — smoke test (a few batches, 1 epoch)

In [ ]:
!python tools/tv_train.py --work-dir {RUNS}/smoke --data-root {DATA_512}/ --no-sas \
    --epochs 1 --batch-size 2 --limit-batches 5
!python tools/tv_eval.py --ckpt {RUNS}/smoke/epoch_1.pth --data-root {DATA_512}/
# GATE: runs end-to-end and prints a finite AP (~0 is expected and fine).

## Phase 5 — RetinaNet baseline (re-run after any time-out; it auto-resumes)

In [ ]:
!python tools/tv_train.py --work-dir {RUNS}/baseline --data-root {DATA_512}/ --no-sas \
    --epochs 24 --batch-size 8 --lr 0.005 --seed 0

In [ ]:
!python tools/tv_eval.py --ckpt {RUNS}/baseline/epoch_24.pth --data-root {DATA_512}/

## Phase 7 — SymFormer w/ RetinaNet (the primary comparison)

In [ ]:
!python tools/tv_train.py --work-dir {RUNS}/symformer --data-root {DATA_512}/ \
    --attention symattention --pe spe --stn --direction r2l \
    --epochs 24 --batch-size 8 --lr 0.005 --seed 0

In [ ]:
!python tools/tv_eval.py --ckpt {RUNS}/symformer/epoch_24.pth --data-root {DATA_512}/
# GATE (primary claim): SymFormer AP50 > baseline AP50. Record both in results.md.

## Phase 8 — Table 8 ablation (long; each cell auto-resumes)
Checkpoints are deleted after each cell is evaluated so Drive stays bounded.

In [ ]:
import subprocess, glob, os
DELETE_CKPTS_AFTER_EVAL = True   # False keeps every ablation checkpoint (~5GB)
EPOCHS = 24

# (name, extra flags) — most-informative cells first, so a partial run is still meaningful.
CELLS = [
    ('none_none',            ['--no-sas']),                                                  # baseline
    ('vanilla_ape',          ['--attention','vanilla','--pe','ape']),
    ('symattention_ape',     ['--attention','symattention','--pe','ape']),
    ('symattention_spe_stn_r2l',   ['--attention','symattention','--pe','spe','--stn','--direction','r2l']),
    ('symattention_spe_nostn_r2l', ['--attention','symattention','--pe','spe','--no-stn','--direction','r2l']),
    ('symattention_spe_stn_l2r',   ['--attention','symattention','--pe','spe','--stn','--direction','l2r']),
    ('vanilla_rpe',          ['--attention','vanilla','--pe','rpe']),
    ('symattention_rpe',     ['--attention','symattention','--pe','rpe']),
    ('vanilla_spe_nostn_l2r',['--attention','vanilla','--pe','spe','--no-stn','--direction','l2r']),
    ('vanilla_spe_nostn_r2l',['--attention','vanilla','--pe','spe','--no-stn','--direction','r2l']),
    ('vanilla_spe_stn_l2r',  ['--attention','vanilla','--pe','spe','--stn','--direction','l2r']),
    ('vanilla_spe_stn_r2l',  ['--attention','vanilla','--pe','spe','--stn','--direction','r2l']),
    ('symattention_spe_nostn_l2r', ['--attention','symattention','--pe','spe','--no-stn','--direction','l2r']),
]

for name, flags in CELLS:
    wd = f'{RUNS}/abl/{name}'
    print('='*70); print('TRAIN', name); print('='*70)
    subprocess.run(['python','tools/tv_train.py','--work-dir',wd,'--data-root',f'{DATA_512}/',
                    '--epochs',str(EPOCHS),'--batch-size','8','--lr','0.005','--seed','0'] + flags)
    ck = f'{wd}/epoch_{EPOCHS}.pth'
    if os.path.isfile(ck):
        print('EVAL', name)
        subprocess.run(['python','tools/tv_eval.py','--ckpt',ck,'--data-root',f'{DATA_512}/',
                        '--tag',name])
        if DELETE_CKPTS_AFTER_EVAL:
            for p in glob.glob(f'{wd}/*.pth'):
                os.remove(p)
            print('removed checkpoints for', name)
    else:
        print('!! no checkpoint for', name, '— training did not finish; re-run this cell to resume')
!du -sh {RUNS}

In [ ]:
# Collect every eval_log.jsonl into one table for results.md
import glob, json
rows = []
for f in sorted(glob.glob(f'{RUNS}/**/eval_log.jsonl', recursive=True)):
    for line in open(f):
        rows.append(json.loads(line))
print(f"{'run':32s} {'AP50':>6s} {'AP':>6s}")
for r in rows:
    print(f"{r['tag']:32s} {r['AP50']:6.1f} {r['AP']:6.1f}")

## Phase 10 — write-up
Fill `results.md` (baseline vs SymFormer + the ablation table), update `CLAUDE.md` §0/§6/§8 with
the actual numbers, and note deviations (torchvision instead of mmdet, batch size, versions) and
caveats (val-not-test; reduced data → trend not absolute numbers). Then commit and push.